In [ ]:
import os
from collections import Counter

path = "E:/DoAn2026/Stickers_Data"
file_extensions = []

for root, dirs, files in os.walk(path):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        file_extensions.append(ext)

print("Thống kê các loại file bạn đang có:")
print(Counter(file_extensions))

In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kết nối DB
conn = psycopg2.connect(
    dbname="KanvaPro",        # Tên DB bạn đã tạo (KanvaPro thay vì your_db)
    user="postgres", 
    password="Superhero3@",  # Mật khẩu bạn dùng trong config Node.js
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("SELECT id FROM asset_categories WHERE slug = 'stickers' LIMIT 1")
res = cur.fetchone()
category_id = res[0] if res else None

source_dir = "E:/DoAn2026/Stickers_Data"

def import_parquet(file_path):
    print(f"--- Đang xử lý file: {os.path.basename(file_path)} ---")
    df = pd.read_parquet(file_path)
    
    # Chuẩn hóa tên cột (Vì mỗi dataset có thể đặt tên cột khác nhau)
    # Giả sử chúng ta cần: name, url, tags
    # Bạn hãy kiểm tra df.columns để mapping cho đúng
    
    data_to_insert = []
    for _, row in df.iterrows():
        # Map dữ liệu từ Parquet sang cấu trúc bảng assets của bạn
        data_to_insert.append((
            row.get('TEXT', 'Unnamed Sticker'), # name
            'sticker',                          # asset_type enum
            row.get('URL', ''),                # url
            row.get('URL', ''),                # thumbnail_url
            0,                                 # file_size (nếu không có)
            row.get('WIDTH', 512),             # width
            row.get('HEIGHT', 512),            # height
            False,                             # is_premium
            category_id,                       # category_id
            ['sticker', 'laion'],              # tags (array)
            '{"source": "LAION-5B"}'           # metadata (jsonb)
        ))

    # Insert theo batch (mỗi lần 1000 dòng để nhanh hơn)
    query = """
        INSERT INTO assets (name, type, url, thumbnail_url, file_size, width, height, is_premium, category_id, tags, metadata)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    execute_batch(cur, query, data_to_insert, page_size=1000)
    conn.commit()
    print(f"Đã xong file {os.path.basename(file_path)}")

# Quét tất cả file .parquet trong các thư mục con
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            import_parquet(os.path.join(root, file))

cur.close()
conn.close()
print("--- HOÀN TẤT TOÀN BỘ ---")

In [ ]:
# %pip install pandas pyarrow psycopg2-binary

In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kết nối DB
conn = psycopg2.connect(
    dbname="KanvaPro", user="postgres", password="Superhero3@", host="localhost"
)
cur = conn.cursor()

# 2. Đảm bảo có Category 'stickers' (Sửa lỗi NoneType tại đây)
cur.execute("""
    INSERT INTO asset_categories (name, slug) 
    VALUES ('Stickers', 'stickers') 
    ON CONFLICT (slug) DO UPDATE SET name = EXCLUDED.name 
    RETURNING id
""")
category_id = cur.fetchone()[0]
conn.commit()

# 3. Đường dẫn lưu file
storage_dir = "E:/DoAn2026/KanvaPro/backend/sticker_upload/assets/stickers"
os.makedirs(storage_dir, exist_ok=True)

source_dir = "E:/DoAn2026/Stickers_Data"

def process_binary_parquet(file_path):
    print(f"--- Đang trích xuất: {os.path.basename(file_path)} ---")
    df = pd.read_parquet(file_path)
    
    data_to_insert = []
    
    for index, row in df.iterrows():
        try:
            # Lấy dữ liệu bytes (Hugging Face thường để trong dict {'bytes': ...})
            img_data = row['image']['bytes']
            desc = row.get('description', 'Pusheen Sticker')
            
            # Tạo tên file: Tên folder + tên file + index để tránh trùng
            parent_folder = os.path.basename(os.path.dirname(os.path.dirname(file_path)))
            filename = f"{parent_folder}_{index}.png"
            local_path = os.path.join(storage_dir, filename)
            
            # Ghi file ảnh
            with open(local_path, 'wb') as f:
                f.write(img_data)
            
            # Đường dẫn mà Frontend sẽ gọi
            internal_url = f"/assets/stickers/{filename}"
            
            data_to_insert.append((
                desc[:200], 'sticker', internal_url, internal_url,
                len(img_data), 512, 512, False, category_id,
                ['sticker', parent_folder, 'pusheen'],
                '{"source": "binary_parquet"}'
            ))
            
        except Exception as e:
            continue

    if data_to_insert:
        query = """
            INSERT INTO assets (name, type, url, thumbnail_url, file_size, width, height, is_premium, category_id, tags, metadata)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        execute_batch(cur, query, data_to_insert, page_size=500)
        conn.commit()
        print(f"Đã lưu {len(data_to_insert)} sticker từ file này.")

# Quét toàn bộ thư mục
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            process_binary_parquet(os.path.join(root, file))

cur.close()
conn.close()
print("--- TẤT CẢ ĐÃ XONG! ---")

In [ ]:
import pandas as pd
import os

source_dir = "E:/DoAn2026/Stickers_Data"

# Tự động tìm file parquet đầu tiên để kiểm tra cấu trúc
found_file = None
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            found_file = os.path.join(root, file)
            break
    if found_file: break

if found_file:
    print(f"--- Đang kiểm tra file: {found_file} ---")
    df = pd.read_parquet(found_file)
    print("\n[DANH SÁCH CỘT THỰC TẾ]:")
    print(df.columns.tolist())
    print("\n[DỮ LIỆU MẪU]:")
    print(df.head(1))
else:
    print("Không tìm thấy bất kỳ file .parquet nào trong thư mục E:/DoAn2026/Stickers_Data")

In [ ]:
import os
import pyarrow.parquet as pq
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kết nối DB với password mới của bạn
try:
    conn = psycopg2.connect(
        dbname="KanvaPro", 
        user="postgres", 
        password="Superhero3@", 
        host="localhost",
        port="5432"
    )
    cur = conn.cursor()
    print("Kết nối Database thành công!")
except Exception as e:
    print(f"Lỗi kết nối DB: {e}")
    exit()

# 2. Lấy hoặc Tạo Category ID (Tránh lỗi NoneType)
cur.execute("""
    INSERT INTO asset_categories (name, slug) 
    VALUES ('Stickers', 'stickers') 
    ON CONFLICT (slug) DO UPDATE SET name = EXCLUDED.name 
    RETURNING id
""")
category_id = cur.fetchone()[0]
conn.commit()

# 3. Cấu hình đường dẫn mới
storage_dir = "E:/DoAn2026/KanvaPro/backend/sticker_upload/assets/stickers"
source_dir = "E:/DoAn2026/Stickers_Data"
os.makedirs(storage_dir, exist_ok=True)

def super_extractor_v3(file_path):
    try:
        # Sử dụng pyarrow để đọc (Tránh lỗi pandas.period extension)
        table = pq.read_table(file_path)
        df = table.to_pandas()
        
        cols = df.columns.tolist()
        
        # Tự động tìm cột chứa ảnh và mô tả
        img_col = next((c for c in cols if any(k in c.lower() for k in ['image', 'img', 'data'])), None)
        txt_col = next((c for c in cols if any(k in c.lower() for k in ['text', 'desc', 'caption', 'title'])), None)
        
        if not img_col:
            return

        data_to_insert = []
        file_basename = os.path.basename(file_path).split('.')[0]

        for index, row in df.iterrows():
            val = row[img_col]
            content = None
            
            # Xử lý các định dạng binary khác nhau trong Parquet
            if isinstance(val, dict) and 'bytes' in val:
                content = val['bytes']
            elif isinstance(val, bytes):
                content = val
            elif hasattr(val, 'get') and val.get('bytes'):
                content = val['bytes']
            
            # Nếu là link URL (str) thì bỏ qua ở script này để tránh treo
            if content is None or isinstance(val, str):
                continue

            # Tạo file name duy nhất
            filename = f"stk_{file_basename}_{index}.png"
            full_local_path = os.path.join(storage_dir, filename)

            with open(full_local_path, 'wb') as f:
                f.write(content)

            name = str(row[txt_col])[:200] if txt_col else "Premium Sticker"
            # Giữ nguyên cấu trúc URL để Frontend/Backend gọi được
            url = f"/assets/stickers/{filename}"
            
            data_to_insert.append((
                name, 'sticker', url, url, len(content), 
                512, 512, False, category_id, 
                ['extracted', 'massive_import'], '{}'
            ))

        if data_to_insert:
            query = """
                INSERT INTO assets (name, type, url, thumbnail_url, file_size, width, height, is_premium, category_id, tags, metadata) 
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            execute_batch(cur, query, data_to_insert, page_size=200)
            conn.commit()
            print(f"Thành công: +{len(data_to_insert)} sticker từ {os.path.basename(file_path)}")
            
    except Exception as e:
        print(f"Lỗi tại {os.path.basename(file_path)}: {str(e)[:100]}")

# 4. Quét toàn bộ thư mục
print("Bắt đầu quét dữ liệu 30GB...")
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            super_extractor_v3(os.path.join(root, file))

cur.close()
conn.close()
print("--- HOÀN TẤT VÉT DỮ LIỆU ---")

In [ ]:
%pip install torch torchvision transformers Pillow

In [ ]:
import torch
print(f"Bản Torch: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")
# Với Series 50, con số này phải từ 9.x hoặc 10.x trở lên

In [6]:
%pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu124

Looking in indexes: https://download.pytorch.org/whl/nightly/cu124
  Using cached https://download.pytorch.org/whl/nightly/cu124/torch-2.7.0.dev20250310%2Bcu124-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/nightly/cu124/torchvision-0.22.0.dev20250226%2Bcu124-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.

The conflict is caused by:
    The user requested torch
    torchvision 0.22.0.dev20250226+cu124 depends on torch==2.7.0.dev20250225+cu124

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

Note: you may need to restart the kernel to use updated packages.


ERROR: Cannot install torch and torchvision==0.22.0.dev20250226+cu124 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
